# Reddit Topic Modeling Pipeline (fastText + KMeans)

This notebook runs the full **end-to-end topic modeling pipeline** on Reddit comments using:

- r/worldnews subset (sampled)
- fastText **unsupervised embeddings** (GeeksForGeeks-style: `train_unsupervised`)
- KMeans clustering
- t-SNE visualization

All core logic is implemented in `reddit_utils.py`.

In [ ]:
# Cell 1: (Optional) Install dependencies
# Skip if already installed in Docker / environment
!pip -q install fasttext nltk pandas numpy scikit-learn matplotlib seaborn tqdm

In [ ]:
# Cell 2: Imports + NLTK setup
import os
import numpy as np
import pandas as pd
import nltk

nltk.download("stopwords")

from reddit_utils import (
    get_english_stopwords,
    load_subreddit_csv,
    write_training_file,
    train_fasttext_unsupervised,
    compute_document_embeddings,
    run_kmeans,
    top_words_by_cluster,
    build_topic_summary,
    run_tsne,
    save_tsne_plot,
)

SEED = 27
np.random.seed(SEED)

In [ ]:
# Cell 3: Provide dataset path (choose ONE option)

# Option A (Docker / local)
# CSV_PATH = "data/reddit_comments.csv"
CSV_PATH = "YOUR_REDDIT_CSV.csv"  # <-- change this

# Option B (Colab)
# from google.colab import files
# uploaded = files.upload()
# CSV_PATH = list(uploaded.keys())[0]

print("Using CSV:", CSV_PATH)

In [ ]:
# Cell 4: Load + filter + sample r/worldnews
stopwords_set = get_english_stopwords()

N_SAMPLES = 10_000
SUBREDDIT = "worldnews"

df = load_subreddit_csv(
    csv_path=CSV_PATH,
    subreddit=SUBREDDIT,
    n_samples=N_SAMPLES,
    random_state=SEED,
    stopwords_set=stopwords_set,
)

print(f"Loaded {len(df)} cleaned comments from r/{SUBREDDIT}")
df.head()

In [ ]:
# Cell 5: Write training text file for fastText
TRAIN_TXT_PATH = "worldnews_training.txt"
TRAIN_TXT_PATH = write_training_file(
    df["clean_comment"].tolist(),
    out_path=TRAIN_TXT_PATH,
)

print("Training file saved to:", TRAIN_TXT_PATH)

with open(TRAIN_TXT_PATH, "r", encoding="utf-8") as f:
    for _ in range(3):
        print(" ", f.readline().strip())

In [ ]:
# Cell 6: Train unsupervised fastText model


FT_MODEL_PATH = "worldnews_fasttext.bin"

ft_model = train_fasttext_unsupervised(
    train_txt_path=TRAIN_TXT_PATH,
    model_type="skipgram",  # or "cbow"
    dim=100,
    epoch=10,
    lr=0.05,
    min_count=2,
    thread=4,
    save_path=FT_MODEL_PATH,
)

print("Trained fastText model. Dimension:", ft_model.get_dimension())
print("Saved model to:", FT_MODEL_PATH)

In [ ]:
# Cell 7: Compute document embeddings
X = compute_document_embeddings(
    ft_model,
    df["clean_comment"].tolist(),
)

print("Embedding matrix shape:", X.shape)

In [ ]:
# Cell 8: KMeans clustering (topics)
N_CLUSTERS = 5

labels, km = run_kmeans(
    X,
    n_clusters=N_CLUSTERS,
    random_state=SEED,
    n_init=10,
)

df["topic"] = labels

print("Topic counts:")
print(df["topic"].value_counts().sort_index())

In [ ]:
# Cell 9: Top words per topic
topic_words = top_words_by_cluster(
    df["clean_comment"].tolist(),
    labels,
    top_n=12,
)

print("\n=== Topic keyword summaries ===")
for t in sorted(topic_words.keys()):
    print(f"Topic {t}: {', '.join(topic_words[t])}")

In [ ]:
# Cell 10: Topic summary table
summary_df = build_topic_summary(
    df,
    top_n_words=10,
    n_examples=3,
)

summary_df

In [ ]:
# Cell 11: t-SNE visualization
TSNE_POINTS = 2000

idx, X_2d = run_tsne(
    X,
    n_points=TSNE_POINTS,
    random_state=SEED,
    perplexity=30,
    learning_rate=200,
    n_iter=1000,
)

labels_for_plot = [
    f"Topic_{t}" for t in df["topic"].iloc[idx].tolist()
]

PLOT_PATH = save_tsne_plot(
    X_2d=X_2d,
    labels=labels_for_plot,
    out_path="tsne_plot.png",
    title=f"t-SNE of r/{SUBREDDIT} topics (fastText + KMeans)",
)

print("Saved t-SNE plot to:", PLOT_PATH)

In [ ]:
# Cell 12: Display the plot
from IPython.display import Image, display

display(Image("tsne_plot.png"))